In [1]:
# Cell 1 — Imports and setup
import basedosdados as bd
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "commodity-credit-risk"
CULTURAS = ("CAFÉ", "CANA-DE-AÇUCAR", "LARANJA")

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery!")
print(f"Project: {PROJECT_ID}")

C:\Users\elemt\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\elemt\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)
C:\Users\elemt\AppData\Local\Pro

Connected to BigQuery!
Project: commodity-credit-risk


In [2]:
# Cell 2 — Query operações de crédito rural (café, cana, laranja)
query = """
SELECT
    op.ano_emissao,
    op.mes_emissao,
    op.data_emissao,
    op.data_vencimento,
    op.sigla_uf,
    op.id_referencia_bacen,
    op.numero_ordem,
    op.id_empreendimento,
    op.id_fonte_recurso,
    op.id_programa,
    op.id_tipo_agricultura,
    op.id_tipo_cultivo,
    op.id_tipo_encargo_financeiro,
    op.valor_parcela_credito,
    op.valor_recurso_proprio,
    op.valor_receita_bruta_esperada,
    op.valor_previsao_producao,
    op.valor_produtividade_obtida,
    op.valor_aliquota_proagro,
    op.taxa_juro,
    op.taxa_juro_encargo_financeiro_posfixado,
    op.valor_percentual_custo_efetivo_total,
    op.valor_percentual_risco_stn,
    emp.produto,
    emp.finalidade,
    emp.modalidade,
    emp.atividade,
    emp.id_tipo_cultura
FROM
    `basedosdados.br_bcb_sicor.operacao` AS op
INNER JOIN
    `basedosdados.br_bcb_sicor.empreendimento` AS emp
ON
    op.id_empreendimento = emp.id_empreendimento
WHERE
    emp.produto IN ('CAFÉ', 'CANA-DE-AÇUCAR', 'LARANJA')
    AND op.ano_emissao BETWEEN 2018 AND 2024
LIMIT 500000
"""

print("Running query...")
df_operacoes = client.query(query).to_dataframe()
print(f"✅ Done! Shape: {df_operacoes.shape}")
df_operacoes.head()


  

Running query...
✅ Done! Shape: (500000, 28)


,ano_emissao,mes_emissao,data_emissao,data_vencimento,sigla_uf,id_referencia_bacen,numero_ordem,id_empreendimento,id_fonte_recurso,id_programa,...,valor_aliquota_proagro,taxa_juro,taxa_juro_encargo_financeiro_posfixado,valor_percentual_custo_efetivo_total,valor_percentual_risco_stn,produto,finalidade,modalidade,atividade,id_tipo_cultura
0,2022,6,2022-06-13,2024-06-15,AC,513976752,1,13121580000002,501,1,...,NaN,0.5,NaN,NaN,NaN,CAFÉ,Investimento,FORMAÇÃO DE CULTURAS PERENES,Agrícola,2
1,2022,6,2022-06-10,2024-06-15,AC,513965023,1,13121840171002,501,1,...,NaN,0.5,NaN,NaN,NaN,CANA-DE-AÇUCAR,Investimento,FORMAÇÃO DE CULTURAS PERENES,Agrícola,2
2,2022,6,2022-06-09,2024-06-15,AC,513962535,1,13121840171002,501,1,...,NaN,0.5,NaN,NaN,NaN,CANA-DE-AÇUCAR,Investimento,FORMAÇÃO DE CULTURAS PERENES,Agrícola,2
3,2022,6,2022-06-10,2024-06-15,AC,513973250,1,13121580000002,501,1,...,NaN,0.5,NaN,NaN,NaN,CAFÉ,Investimento,FORMAÇÃO DE CULTURAS PERENES,Agrícola,2
4,2022,6,2022-06-28,2032-07-10,AC,514082304,1,13121580000002,501,1,...,NaN,0.5,NaN,NaN,NaN,CAFÉ,Investimento,FORMAÇÃO DE CULTURAS PERENES,Agrícola,2


In [4]:
# Cell 3 — Save to SQLite
import sqlite3
import os

os.makedirs("../data", exist_ok=True)
DB_PATH = "../data/commodity_credit.db"

conn = sqlite3.connect(DB_PATH)
df_operacoes.to_sql("operacoes", conn, if_exists="replace", index=False)
conn.close()

print(f"✅ Saved to SQLite: {DB_PATH}")
print(f"Rows saved: {len(df_operacoes):,}")

✅ Saved to SQLite: ../data/commodity_credit.db
Rows saved: 500,000


In [7]:
# Cell 4 — Query tabela saldo (inadimplência)
query_saldo = """
SELECT
    s.ano,
    s.mes,
    s.ano_emissao,
    s.mes_emissao,
    s.id_referencia_bacen,
    s.numero_ordem,
    s.id_situacao_operacao,
    s.valor_medio_diario,
    s.valor_medio_diario_vincendo,
    s.valor_ultimo_dia
FROM
    `basedosdados.br_bcb_sicor.saldo` AS s
INNER JOIN
    `basedosdados.br_bcb_sicor.operacao` AS op
ON
    s.id_referencia_bacen = op.id_referencia_bacen
    AND s.numero_ordem = op.numero_ordem
INNER JOIN
    `basedosdados.br_bcb_sicor.empreendimento` AS emp
ON
    op.id_empreendimento = emp.id_empreendimento
WHERE
    emp.produto IN ('CAFÉ', 'CANA-DE-AÇUCAR', 'LARANJA')
    AND s.ano BETWEEN 2018 AND 2024
LIMIT 500000
"""

print("Running query saldo...")
df_saldo = client.query(query_saldo).to_dataframe()
print(f"✅ Done! Shape: {df_saldo.shape}")
print(f"\nSituações de operação:")
print(df_saldo['id_situacao_operacao'].value_counts())

Running query saldo...
✅ Done! Shape: (500000, 10)

Situações de operação:
id_situacao_operacao
1     420211
7      24273
3      17839
4      14014
12     10903
2       6463
9       6029
6        216
13        20
5         20
8         12
Name: count, dtype: int64


In [8]:
# Check situacao operacao meanings
query_situacao = """
SELECT DISTINCT
    id_situacao_operacao,
    COUNT(*) as total
FROM `basedosdados.br_bcb_sicor.saldo`
WHERE ano BETWEEN 2018 AND 2024
GROUP BY id_situacao_operacao
ORDER BY total DESC
"""

df_situacao = client.query(query_situacao).to_dataframe()
print(df_situacao.to_string())

   id_situacao_operacao      total
0                     1  304740609
1                    12   16604835
2                     3    9317273
3                     4    7859414
4                     7    7136122
5                     2    5507559
6                     9    2879239
7                     6      99532
8                     5      40802
9                     8      13310
10                   13      12933
11                   11        355
12                   10        180


In [10]:
# Check dicionario table
query_dic = """
SELECT *
FROM `basedosdados.br_bcb_sicor.dicionario`
WHERE id_tabela = 'saldo'
AND nome_coluna = 'id_situacao_operacao'
ORDER BY chave
"""

df_dic = client.query(query_dic).to_dataframe()
print(df_dic.to_string())

   id_tabela           nome_coluna chave cobertura_temporal                                       valor
0      saldo  id_situacao_operacao     1                (1)                                Curso normal
1      saldo  id_situacao_operacao    10                (1)                                    Excluída
2      saldo  id_situacao_operacao    11                (1)           Inscrita em Dívida Ativa da União
3      saldo  id_situacao_operacao    12                (1)                                Inadimplente
4      saldo  id_situacao_operacao    13                (1)                Desclassificada Parcialmente
5      saldo  id_situacao_operacao     2                (1)                                   Em atraso
6      saldo  id_situacao_operacao     3                (1)                                  Prorrogada
7      saldo  id_situacao_operacao     4                (1)               Renegociada Sem Nova Operação
8      saldo  id_situacao_operacao     5                (1)  Ren

In [11]:
# Cell 5 — Save saldo and dicionario to SQLite
conn = sqlite3.connect(DB_PATH)

df_saldo.to_sql("saldo", conn, if_exists="replace", index=False)
df_dic.to_sql("dicionario_situacao", conn, if_exists="replace", index=False)

# Save situacao mapping as reference
situacao_map = {
    1: "Curso normal",
    2: "Em atraso",
    3: "Prorrogada",
    4: "Renegociada Sem Nova Operação",
    5: "Renegociada Parcialmente Com Nova Operação",
    6: "Renegociada Totalmente Com Nova Operação",
    7: "Liquidada",
    8: "Desclassificada",
    9: "Baixada como Prejuízo",
    10: "Excluída",
    11: "Inscrita em Dívida Ativa da União",
    12: "Inadimplente",
    13: "Desclassificada Parcialmente"
}

import pandas as pd
df_situacao_map = pd.DataFrame(
    list(situacao_map.items()),
    columns=["id_situacao_operacao", "descricao"]
)
df_situacao_map.to_sql("situacao_map", conn, if_exists="replace", index=False)

conn.close()

print(f"✅ Saved to SQLite!")
print(f"  operacoes: {len(df_operacoes):,} rows")
print(f"  saldo: {len(df_saldo):,} rows")
print(f"  dicionario_situacao: {len(df_dic)} rows")
print(f"  situacao_map: {len(df_situacao_map)} rows")

✅ Saved to SQLite!
  operacoes: 500,000 rows
  saldo: 500,000 rows
  dicionario_situacao: 13 rows
  situacao_map: 13 rows


In [12]:
# Cell 6 — Database integrity check
conn = sqlite3.connect(DB_PATH)

print("=== DATABASE SUMMARY ===\n")

tables = ["operacoes", "saldo", "situacao_map", "dicionario_situacao"]
for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0,0]
    print(f"  {table}: {count:,} rows")

print("\n=== OPERACOES — produtos ===")
print(pd.read_sql("""
    SELECT produto, COUNT(*) as total, 
           ROUND(AVG(valor_parcela_credito), 2) as ticket_medio
    FROM operacoes 
    GROUP BY produto 
    ORDER BY total DESC
""", conn).to_string(index=False))

print("\n=== SALDO — situacoes ===")
print(pd.read_sql("""
    SELECT sm.descricao, COUNT(*) as total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM saldo s
    JOIN situacao_map sm ON CAST(s.id_situacao_operacao AS INTEGER) = sm.id_situacao_operacao
    GROUP BY sm.descricao
    ORDER BY total DESC
""", conn).to_string(index=False))

conn.close()

=== DATABASE SUMMARY ===

  operacoes: 500,000 rows
  saldo: 500,000 rows
  situacao_map: 13 rows
  dicionario_situacao: 13 rows

=== OPERACOES — produtos ===
       produto  total  ticket_medio
          CAFÉ 388107     197152.13
CANA-DE-AÇUCAR  87522     517058.89
       LARANJA  24371     313043.10

=== SALDO — situacoes ===
                                 descricao  total   pct
                              Curso normal 420211 84.04
                                 Liquidada  24273  4.85
                                Prorrogada  17839  3.57
             Renegociada Sem Nova Operação  14014  2.80
                              Inadimplente  10903  2.18
                                 Em atraso   6463  1.29
                     Baixada como Prejuízo   6029  1.21
  Renegociada Totalmente Com Nova Operação    216  0.04
              Desclassificada Parcialmente     20  0.00
Renegociada Parcialmente Com Nova Operação     20  0.00
                           Desclassificada     12  0.0